# 002 - CatBoost baseline (short-horizon)

Цель: честно обучить первую baseline-модель CatBoost на уже собранном `model_dataset` и оценить качество (ML + trading-oriented) без leakage.

Ноутбук — оркестратор: загрузка данных, выбор target/фич, time-based split, обучение, отчёт и сохранение артефактов.


In [2]:
import sys
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd

# Root detection (чтобы запускать ноутбук из любой директории)
_cwd = Path.cwd().resolve()
_repo_root_candidates = [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]

_repo_root = None
for c in _repo_root_candidates:
    if (c / "research").exists() and (c / "notebooks").exists():
        _repo_root = c
        break

if _repo_root is None:
    raise RuntimeError("Could not detect repo root")

sys.path.insert(0, str(_repo_root))
print("root:", _repo_root)


root: D:\tumar\okx-hft-research


In [3]:
# ----------------------------
# Config
# ----------------------------
RUN_TAG = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

DATASET_METADATA_PATH = _repo_root / (
    "data/processed/modeling/btc_usdt_swap_model_dataset_100ms_baseline_20260323_000000__20260325_235959_metadata.json"
)

DATASET_PARQUET_PATH = DATASET_METADATA_PATH.with_name(
    DATASET_METADATA_PATH.name.replace("_metadata.json", ".parquet")
)

# Baseline target priority for horizon=100ms
HORIZON_MS = 100
TARGET_PRIORITY = ("net", "gross", "mid_dir")

# Time split (must be time-based, not random)
TRAIN_FRAC = 0.70
VALID_FRAC = 0.15

# CatBoost hyperparameters (reasonable baseline)
CAT_ITERATIONS = 4000
CAT_LEARNING_RATE = 0.03
CAT_DEPTH = 8
CAT_L2_LEAF_REG = 3.0
CAT_EARLY_STOPPING_ROUNDS = 200
CAT_RANDOM_SEED = 42
CAT_VERBOSE = 200

# Optional: limit rows for quick local iteration (set to None for full run)
MAX_ROWS_TOTAL = None

# Confidence thresholds for bucket evaluation
CONF_THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

PREDICTIONS_TOP_N = 50_000

print("dataset parquet exists:", DATASET_PARQUET_PATH.exists())
print("dataset metadata exists:", DATASET_METADATA_PATH.exists())


dataset parquet exists: True
dataset metadata exists: True


C:\Users\sorok\AppData\Local\Temp\ipykernel_29228\3294807419.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TAG = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [4]:
from research.models.catboost_baseline_helpers import (
    SplitConfig,
    TrainingConfig,
    build_feature_columns,
    time_split,
    time_split_boundaries,
    make_class_distribution_report,
    infer_target_type,
    normalize_multiclass_labels,
    compute_binary_class_weights,
    train_catboost_binary,
    train_catboost_multiclass,
    evaluate_binary_classifier,
    evaluate_multiclass_classifier,
    make_confidence_bucket_report,
    trading_evaluation_profitability,
    reliability_table,
    get_feature_importance,
    save_predictions_parquet,
    calibration_curve_data,
)

assert DATASET_METADATA_PATH.exists()
assert DATASET_PARQUET_PATH.exists()

metadata: dict = json.loads(DATASET_METADATA_PATH.read_text(encoding="utf-8"))
print("Loaded metadata")
print("feature_cols:", len(metadata.get("feature_cols", [])))
print("targets (profitability):", len(metadata.get("targets", {}).get("profitability", [])))


Loaded metadata
feature_cols: 110
targets (profitability): 36


In [5]:
# Load dataset (only needed columns)
feature_cols_from_meta = metadata.get("feature_cols", [])

# Candidate target columns in priority order
candidate_targets = []
for p in TARGET_PRIORITY:
    if p == "net":
        candidate_targets.append(f"target_net_profitable_h{HORIZON_MS}")
    elif p == "gross":
        candidate_targets.append(f"target_gross_profitable_h{HORIZON_MS}")
    elif p == "mid_dir":
        candidate_targets.append(f"target_mid_dir_h{HORIZON_MS}")
    else:
        raise ValueError(f"Unknown priority element: {p}")

columns_needed = list(dict.fromkeys(feature_cols_from_meta + candidate_targets))
df = pd.read_parquet(DATASET_PARQUET_PATH, columns=columns_needed)

selected_target_col = None
for c in candidate_targets:
    if c in df.columns:
        selected_target_col = c
        break

if selected_target_col is None:
    raise RuntimeError(f"None of candidate targets found: {candidate_targets}")

target_col = selected_target_col
target_type = infer_target_type(df[target_col])

if MAX_ROWS_TOTAL is not None:
    df = df.iloc[:MAX_ROWS_TOTAL].copy()

print("Selected target:", target_col)
print("Target type:", target_type)
print("Loaded df rows:", len(df))
print("Columns loaded:", len(df.columns))


Selected target: target_net_profitable_h100
Target type: binary
Loaded df rows: 2592052
Columns loaded: 113


In [6]:
# Build leakage-safe feature set
used_feature_columns, excluded_reasons = build_feature_columns(
    df=df, metadata=metadata, target_col=target_col
)

excluded_df = pd.DataFrame(excluded_reasons).sort_values(["reason", "column"])
used_feature_columns = list(used_feature_columns)

print("Used feature columns:", len(used_feature_columns))
print("Excluded columns:", len(excluded_reasons))

excluded_df.head(20)


Used feature columns: 105
Excluded columns: 5


,column,reason
2,anchor_snapshot_ts,identifier / time column
1,inst_id,identifier / time column
3,reconstruction_mode,identifier / time column
4,tick_size,identifier / time column
0,ts_event,identifier / time column


## Leakage checks

Группы колонок, которые НЕ попадают в модель:

- `target_*` и `label_*`: целевая разметка (не использовать как признаки)
- `mid_move_ticks_*`, `best_bid_move_ticks_*`, `best_ask_move_ticks_*`: будущие move-производные признаки
- `ts_event`, `inst_id`, `anchor_snapshot_ts`, `reconstruction_mode`, `tick_size`: служебные/технические поля

Time split:
- split по отсортированному `ts_event` (без random shuffle), чтобы исключить leakage по времени.


In [7]:
# Prepare supervised arrays
# Important: don't drop rows for NaNs in features; CatBoost can handle NaNs.
df = df.dropna(subset=[target_col, "ts_event"]).copy()
df = df.sort_values("ts_event").reset_index(drop=True)
print("Rows after dropna(target/ts_event):", len(df))
if len(df) == 0:
    raise ValueError(
        "Dataset became empty after dropna(target/ts_event). "
        "This usually means: target column is extremely sparse or contains too many NaNs. "
        "Switch TARGET_PRIORITY in Config (e.g. to target_gross_profitable_h100 or target_mid_dir_h100)."
    )

split_cfg = SplitConfig(
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    test_frac=1 - TRAIN_FRAC - VALID_FRAC,
)
bounds = time_split_boundaries(df, ts_col="ts_event", train_frac=TRAIN_FRAC, valid_frac=VALID_FRAC)
splits = time_split(df, ts_col="ts_event", target_col=target_col, cfg=split_cfg)

print("Split boundaries:", bounds)
print("Rows:", {k: len(v) for k, v in splits.items()})

dist_train = make_class_distribution_report(splits["train"], target_col)
dist_valid = make_class_distribution_report(splits["valid"], target_col)
dist_test = make_class_distribution_report(splits["test"], target_col)

print("Train dist:", dist_train)
print("Valid dist:", dist_valid)
print("Test dist:", dist_test)

if target_type == "binary":
    y_train_raw = splits["train"][target_col].astype(int).to_numpy()
    y_valid_raw = splits["valid"][target_col].astype(int).to_numpy()
    y_test_raw = splits["test"][target_col].astype(int).to_numpy()

    uniq = sorted(list(np.unique(y_train_raw)))
    print("Binary unique labels:", uniq)

    if set(uniq) == {0, 1}:
        y_train = y_train_raw
        y_valid = y_valid_raw
        y_test = y_test_raw
    elif set(uniq) == {-1, 1}:
        y_train = (y_train_raw > 0).astype(int)
        y_valid = (y_valid_raw > 0).astype(int)
        y_test = (y_test_raw > 0).astype(int)
    else:
        # last resort: map min->0, max->1
        mn, mx = min(uniq), max(uniq)
        y_train = (y_train_raw == mx).astype(int)
        y_valid = (y_valid_raw == mx).astype(int)
        y_test = (y_test_raw == mx).astype(int)

    pos_frac = float(y_train.mean())
    imbalance_ratio = float((1 - pos_frac) / max(pos_frac, 1e-12))
    print("Positive fraction (train):", pos_frac)
    print("Imbalance ratio (neg/pos):", imbalance_ratio)


Rows after dropna(target/ts_event): 2592052
Split boundaries: {'n': 2592052, 'train_end_idx': 1814436, 'valid_end_idx': 2203244, 'train': {'start': Timestamp('2026-03-23 00:00:00+0000', tz='UTC'), 'end': Timestamp('2026-03-25 02:23:58.500000+0000', tz='UTC')}, 'valid': {'start': Timestamp('2026-03-25 02:23:58.600000+0000', tz='UTC'), 'end': Timestamp('2026-03-25 13:11:58.200000+0000', tz='UTC')}, 'test': {'start': Timestamp('2026-03-25 13:11:58.300000+0000', tz='UTC'), 'end': Timestamp('2026-03-25 23:59:58+0000', tz='UTC')}}
Rows: {'train': 1814436, 'valid': 388808, 'test': 388808}
Train dist: {'counts': {0: 1814400, 1: 36}, 'shares': {'0': 0.9999801591238269, '1': 1.9840876173091802e-05}, 'n': 1814436}
Valid dist: {'counts': {0: 388807, 1: 1}, 'shares': {'0': 0.9999974280364602, '1': 2.571963539844859e-06}, 'n': 388808}
Test dist: {'counts': {0: 388797, 1: 11}, 'shares': {'0': 0.9999717084010618, '1': 2.829159893829345e-05}, 'n': 388808}
Binary unique labels: [np.int64(0), np.int64(1)

In [8]:
# Build X matrices
X_train = splits["train"][used_feature_columns]
X_valid = splits["valid"][used_feature_columns]
X_test = splits["test"][used_feature_columns]

train_params = TrainingConfig(
    iterations=CAT_ITERATIONS,
    learning_rate=CAT_LEARNING_RATE,
    depth=CAT_DEPTH,
    l2_leaf_reg=CAT_L2_LEAF_REG,
    random_seed=CAT_RANDOM_SEED,
    early_stopping_rounds=CAT_EARLY_STOPPING_ROUNDS,
    verbose=CAT_VERBOSE,
)

catboost_params_used: dict = {
    "iterations": train_params.iterations,
    "learning_rate": train_params.learning_rate,
    "depth": train_params.depth,
    "l2_leaf_reg": train_params.l2_leaf_reg,
    "random_seed": train_params.random_seed,
    "early_stopping_rounds": train_params.early_stopping_rounds,
}


In [ ]:
# Train CatBoost baseline
model = None

if target_type == "binary":
    pos_ratio = float(y_train.mean())
    use_class_weights = pos_ratio < 0.30 and pos_ratio > 0.0
    class_weights = None
    if use_class_weights:
        class_weights = compute_binary_class_weights(y_train)
    print("Using class_weights:", class_weights)

    model = train_catboost_binary(
        X_train,
        y_train,
        X_valid,
        y_valid,
        train_params=train_params,
        class_weights=class_weights,
    )
else:
    y_train_mapped, mapping = normalize_multiclass_labels(splits["train"][target_col])
    y_valid_mapped, _ = normalize_multiclass_labels(splits["valid"][target_col])
    y_test_mapped, _ = normalize_multiclass_labels(splits["test"][target_col])
    num_classes = int(len(mapping))
    print("Multiclass mapping:", mapping)

    inv_mapping = {v: k for k, v in mapping.items()}
    class_names = [str(inv_mapping[i]) for i in range(num_classes)]

    model = train_catboost_multiclass(
        X_train,
        y_train_mapped,
        X_valid,
        y_valid_mapped,
        train_params=train_params,
        num_classes=num_classes,
    )


Using class_weights: {0: 0.5000099206349207, 1: 25200.5}
0:	test: 0.5837086	best: 0.5837086 (0)	total: 874ms	remaining: 58m 16s


In [ ]:
# Evaluate
if target_type == "binary":
    valid_metrics, test_metrics = evaluate_binary_classifier(
        model=model,
        X_valid=X_valid,
        y_valid=y_valid,
        X_test=X_test,
        y_test=y_test,
    )

    y_proba_valid = model.predict_proba(X_valid)[:, 1]
    y_proba_test = model.predict_proba(X_test)[:, 1]

    confidence_valid = make_confidence_bucket_report(
        y_true=y_valid,
        y_proba_pos=y_proba_valid,
        thresholds=CONF_THRESHOLDS,
    )
    confidence_test = make_confidence_bucket_report(
        y_true=y_test,
        y_proba_pos=y_proba_test,
        thresholds=CONF_THRESHOLDS,
    )

    calibration_table = reliability_table(y_test, y_proba_test, n_bins=10)
    calibration_curve_df = calibration_curve_data(y_test, y_proba_test, n_bins=10)

    y_pred_test = (y_proba_test >= 0.5).astype(int)
else:
    # Remap using the same normalization approach
    y_train_mapped, mapping = normalize_multiclass_labels(splits["train"][target_col])
    y_valid_mapped, _ = normalize_multiclass_labels(splits["valid"][target_col])
    y_test_mapped, _ = normalize_multiclass_labels(splits["test"][target_col])

    inv_mapping = {v: k for k, v in mapping.items()}
    num_classes = len(mapping)
    class_names = [str(inv_mapping[i]) for i in range(num_classes)]

    valid_metrics, test_metrics, confidence_test = evaluate_multiclass_classifier(
        model=model,
        X_valid=X_valid,
        y_valid=y_valid_mapped,
        X_test=X_test,
        y_test=y_test_mapped,
        class_names=class_names,
        confidence_thresholds=CONF_THRESHOLDS,
    )


In [ ]:
print("Validation metrics:")
print(json.dumps(valid_metrics, ensure_ascii=False, indent=2)[:4000])
print("Test metrics:")
print(json.dumps(test_metrics, ensure_ascii=False, indent=2)[:4000])


In [ ]:
# Feature importance
feature_importance_df = get_feature_importance(model, used_feature_columns, top_n=30)
feature_importance_df.head(15)


In [ ]:
# Trading-oriented evaluation (only for profitability-style binary targets)
trading_report_df = None
if "profitable" in target_col and target_type == "binary":
    trading_report_df = trading_evaluation_profitability(
        y_true=y_test,
        y_proba_pos=y_proba_test,
        thresholds=CONF_THRESHOLDS,
    )
    trading_report_df
else:
    print("Trading eval skipped: target is not profitability-binary or target_type != binary")


In [ ]:
# Save artifacts
artifacts_models_dir = _repo_root / "data/artifacts/models"
artifacts_reports_dir = _repo_root / "data/artifacts/reports"
artifacts_models_dir.mkdir(parents=True, exist_ok=True)
artifacts_reports_dir.mkdir(parents=True, exist_ok=True)

run_id = f"catboost_baseline_{target_col}_h{HORIZON_MS}_{RUN_TAG}"
run_models_dir = artifacts_models_dir / run_id
run_models_dir.mkdir(parents=True, exist_ok=True)

model_path = run_models_dir / "model.cbm"
pred_path = run_models_dir / "predictions_test.parquet"
validation_metrics_path = run_models_dir / "validation_metrics.json"
test_metrics_path = run_models_dir / "test_metrics.json"
confidence_test_path = run_models_dir / "confidence_buckets_test.csv"
calibration_path = run_models_dir / "calibration_bins_test.csv"
trading_report_path = run_models_dir / "trading_eval_test.csv"
feature_importance_path = run_models_dir / "feature_importance_top30.csv"
metadata_out_path = run_models_dir / "model_metadata.json"
report_md = artifacts_reports_dir / f"{run_id}.md"

# Save model
model.save_model(str(model_path))

if target_type == "binary":
    save_predictions_parquet(
        pred_path,
        ts_event=splits["test"]["ts_event"],
        y_true=y_test,
        y_pred=y_pred_test,
        y_proba_pos=y_proba_test,
    )
    confidence_test.to_csv(confidence_test_path, index=False)
    calibration_table.to_csv(calibration_path, index=False)
    if trading_report_df is not None:
        trading_report_df.to_csv(trading_report_path, index=False)
else:
    proba_test = model.predict_proba(X_test)
    y_pred_test = proba_test.argmax(axis=1)
    y_proba_max_test = proba_test.max(axis=1)
    out = pd.DataFrame(
        {
            "ts_event": splits["test"]["ts_event"].values,
            "y_true": y_test_mapped.astype(int),
            "y_pred": y_pred_test.astype(int),
            "y_proba_max": y_proba_max_test.astype(float),
        }
    )
    out.to_parquet(pred_path, index=False)
    confidence_test.to_csv(confidence_test_path, index=False)

feature_importance_df.to_csv(feature_importance_path, index=False)

validation_metrics_path.write_text(
    json.dumps(valid_metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
test_metrics_path.write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)

split_info = {
    "boundaries": bounds,
    "row_counts": {k: int(len(v)) for k, v in splits.items()},
    "target_distribution": {
        "train": dist_train,
        "valid": dist_valid,
        "test": dist_test,
    },
}

metadata_out = {
    "run_id": run_id,
    "timestamp_utc": RUN_TAG,
    "dataset_parquet_path": str(DATASET_PARQUET_PATH),
    "dataset_metadata_path": str(DATASET_METADATA_PATH),
    "target_col": target_col,
    "target_type": target_type,
    "horizon_ms": HORIZON_MS,
    "used_feature_columns": used_feature_columns,
    "excluded_columns_with_reasons": excluded_reasons,
    "split": split_info,
    "catboost_train_params": catboost_params_used,
    "validation_metrics": valid_metrics,
    "test_metrics": test_metrics,
    "artifacts": {
        "model_path": str(model_path),
        "predictions_test_path": str(pred_path),
        "confidence_test_path": str(confidence_test_path),
        "calibration_path": str(calibration_path),
        "trading_report_path": str(trading_report_path),
        "feature_importance_path": str(feature_importance_path),
        "validation_metrics_path": str(validation_metrics_path),
        "test_metrics_path": str(test_metrics_path),
    },
}
metadata_out_path.write_text(
    json.dumps(metadata_out, ensure_ascii=False, indent=2), encoding="utf-8"
)

report_md.write_text(
    "\n".join(
        [
            f"# {run_id}",
            f"Target: `{target_col}` ({target_type}), horizon={HORIZON_MS}ms",
            "",
            "## Key metrics (validation)",
            f"```json\n{json.dumps(valid_metrics, ensure_ascii=False, indent=2)}\n```",
            "",
            "## Key metrics (test)",
            f"```json\n{json.dumps(test_metrics, ensure_ascii=False, indent=2)}\n```",
            "",
            "## Top features",
            f"Saved: `{feature_importance_path}`",
            "",
            "## Artifacts",
            f"Model: `{model_path}`",
            f"Predictions (test): `{pred_path}`",
        ]
    ),
    encoding="utf-8",
)

print("Model saved:", model_path)
print("Predictions saved:", pred_path)
print("Metadata saved:", metadata_out_path)
print("Report saved:", report_md)


## Вывод

1) Если ключевые метрики на `test` слабые и/или high-confidence buckets не подтверждаются — baseline сигнал слабый.
2) Если качество стабильно держится на `test` — это честная основа для следующего шага (другой target/horizon или более сложная модель).
